# Video Pipeline

This notebook demonstrates docling's video processing pipeline, which:
- Transcribes audio using Whisper ASR
- Samples representative frames using scene-change detection
- Optionally assigns speaker labels via diarization
- Exports to HTML, Markdown, JSON, or WebVTT

**Use cases:**
- Business meeting recordings → searchable transcript with frames
- Lecture videos → structured notes with slide captures
- Any video → subtitle file (.vtt) played back with captions in a browser


## Setup

Install docling with video support (ASR + speaker diarization):
```bash
pip install 'docling-slim[format-video]'
```

FFmpeg must also be installed on your system:
```bash
# macOS
brew install ffmpeg
# Ubuntu/Debian
sudo apt-get install ffmpeg
```

In [1]:
from pathlib import Path

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import VideoPipelineOptions
from docling.document_converter import DocumentConverter, VideoFormatOption
from docling.utils.video_frame_sampling import VideoFrameSamplingMode


def print_transcript(document):
    """Print each transcript segment as [mm:ss] [speaker] text."""
    for item, _ in document.iterate_items():
        if not (hasattr(item, "text") and item.text):
            continue
        track = item.source[0] if item.source else None
        ts = track.start_time if track else 0.0
        speaker = f" [{track.voice}]" if track and track.voice else ""
        m, s = divmod(int(ts), 60)
        print(f"[{m:02d}:{s:02d}]{speaker} {item.text.strip()}")

## Basic Usage

Convert a video using fixed-interval frame sampling (one frame every 10 seconds):

In [2]:
VIDEO_PATH = Path("data/single_person_single_scene_lf.mp4")  # replace with your video

dc = DocumentConverter(allowed_formats=[InputFormat.VIDEO])
result = dc.convert(VIDEO_PATH)

print(f"Status: {result.status}")
print(f"Transcript segments: {len(result.document.texts)}")
print(f"Frames captured: {len(result.document.pictures)}")

100%|█████████████████████████████████████| 72.1M/72.1M [00:10<00:00, 7.48MiB/s]
/home/neebal/Desktop/docling/.venv/lib/python3.12/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Detecting language using up to the first 30 seconds. Use `--language` to specify the language
Detected language: English
[00:00.000 --> 00:08.460]  You were one of the three authors with Alex Kashowski, Jeff Hinton of the famed Alex Ned paper that is arguably
Status: ConversionStatus.SUCCESS
Transcript segments: 1
Frames captured: 1


## Scene-Change Detection

For meeting recordings, scene-change sampling captures frames at meaningful transitions
rather than at fixed intervals. The `prominence` parameter controls sensitivity —
lower values detect more subtle scene changes.

**Recommended settings:**
- Business meetings: `prominence=0.03`
- Lectures with slides: `cuts_per_minute=2`
- Dynamic content: `prominence=0.01`

In [4]:
VIDEO_PATH = Path("data/two_person_multi_scene_lf_is.mp4")  # replace with your video

opts = VideoPipelineOptions(
    frame_sampling_mode=VideoFrameSamplingMode.SCENE_CHANGE,
    scene_change_prominence=0.03,  # recommended for meetings
    min_scene_duration_seconds=2.0,
)

dc = DocumentConverter(
    allowed_formats=[InputFormat.VIDEO],
    format_options={InputFormat.VIDEO: VideoFormatOption(pipeline_options=opts)},
)
result = dc.convert(VIDEO_PATH)

print_transcript(result.document)

/home/neebal/Desktop/docling/.venv/lib/python3.12/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Detecting language using up to the first 30 seconds. Use `--language` to specify the language
Detected language: English
[00:00.000 --> 00:08.620]  You were one of the three authors with Alex Kishowski, Jeff Hinton of the famed Alex Ned paper that is arguably
[00:09.260 --> 00:13.800]  the paper that marked a big catalytic moment that launched a deep learning revolution
[00:14.540 --> 00:17.800]  At that time take us back to that time. What was your intuition about
[00:18.380 --> 00:22.060]  Neon networks about the representational power of Neon networks?
[00:23.480 --> 00:23.500]  And
[00:23.500 --> 00:30.000]  maybe you can imagine how did that evolve over the next few years up to today over the ten years?
[00:30.740 --> 00:38.780]  Yeah, I can answer that question. At some point in about 2011, I connected two facts in my mind.
[00:39.860 --> 00:47.340]  Basically, the realization was this. At some point, we realized that we can train very large
[00:47.440 --> 00:53.100]  I shouldn't

## Speaker Diarization

Enable speaker diarization to identify who is speaking in each segment.
Requires `resemblyzer` and `scikit-learn`.
Speaker count is automatically detected.

In [5]:
opts = VideoPipelineOptions(
    frame_sampling_mode=VideoFrameSamplingMode.SCENE_CHANGE,
    scene_change_prominence=0.03,
    enable_diarization=True,  # requires resemblyzer
)

dc = DocumentConverter(
    allowed_formats=[InputFormat.VIDEO],
    format_options={InputFormat.VIDEO: VideoFormatOption(pipeline_options=opts)},
)
result = dc.convert(VIDEO_PATH)

print_transcript(result.document)

/home/neebal/Desktop/docling/.venv/lib/python3.12/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Detecting language using up to the first 30 seconds. Use `--language` to specify the language
Detected language: English
[00:00.000 --> 00:08.620]  You were one of the three authors with Alex Kishowski, Jeff Hinton of the famed Alex Ned paper that is arguably
[00:09.260 --> 00:13.800]  the paper that marked a big catalytic moment that launched a deep learning revolution
[00:14.540 --> 00:17.800]  At that time take us back to that time. What was your intuition about
[00:18.380 --> 00:22.060]  Neon networks about the representational power of Neon networks?
[00:23.480 --> 00:23.500]  And
[00:23.500 --> 00:30.000]  maybe you can imagine how did that evolve over the next few years up to today over the ten years?
[00:30.740 --> 00:38.780]  Yeah, I can answer that question. At some point in about 2011, I connected two facts in my mind.
[00:39.860 --> 00:47.340]  Basically, the realization was this. At some point, we realized that we can train very large
[00:47.440 --> 00:53.100]  I shouldn't

/home/neebal/Desktop/docling/.venv/lib/python3.12/site-packages/webrtcvad.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Loaded the voice encoder model on cpu in 0.03 seconds.
[00:00] [SPEAKER_01] You were one of the three authors with Alex Kishowski, Jeff Hinton of the famed Alex Ned paper that is arguably the paper that marked a big catalytic moment that launched a deep learning revolution At that time take us back to that time. What was your intuition about Neon networks about the representational power of Neon networks?
[00:23] [SPEAKER_01] And maybe you can imagine how did that evolve over the next few years up to today over the ten years?
[00:30] [SPEAKER_00] Yeah, I can answer that question. At some point in about 2011, I connected two facts in my mind.
[00:39] [SPEAKER_00] Basically, the realization was this. At some point, we realized that we can train very large I shouldn't say very in other tiny by today's standards, but large and deep neural networks end to end with back propagation.
[00:55] [SPEAKER_00] At some point, different people obtained this result, I obtained this result.
[01:01] [SP

## Export Formats

The `DoclingDocument` produced by the video pipeline can be exported to any format
that docling supports.

In [6]:
output_dir = Path("scratch")
output_dir.mkdir(exist_ok=True)

# HTML — transcript with timestamps and embedded frames
result.document.save_as_html(output_dir / "video.html")

# Markdown — plain transcript
result.document.save_as_markdown(output_dir / "video.md")

# JSON — full structured document
result.document.save_as_json(output_dir / "video.json")

# WebVTT — subtitle file
result.document.save_as_vtt(output_dir / "video.vtt")

print("Exported to:", list(output_dir.iterdir()))

Exported to: [PosixPath('scratch/2305.03393v1-pg9-deepseek_ocr-api.html'), PosixPath('scratch/video.md'), PosixPath('scratch/2305.03393v1-pg9-granite_vision-transformers.md'), PosixPath('scratch/2305.03393v1-pg9-nanonets_ocr2-transformers.html'), PosixPath('scratch/video.html'), PosixPath('scratch/2305.03393v1-pg9-granite_vision-transformers.json'), PosixPath('scratch/video.vtt'), PosixPath('scratch/2305.03393v1-pg9-nanonets_ocr2-transformers.json'), PosixPath('scratch/1G'), PosixPath('scratch/2305.03393v1-pg9-granite_docling-transformers.json'), PosixPath('scratch/video.json'), PosixPath('scratch/2305.03393v1-pg9-deepseek_ocr-api.md'), PosixPath('scratch/2305.03393v1-pg9-granite_docling-transformers.md'), PosixPath('scratch/2305.03393v1-pg9-granite_docling-transformers.html'), PosixPath('scratch/1DA'), PosixPath('scratch/2305.03393v1-pg9-deepseek_ocr-api.json'), PosixPath('scratch/1B'), PosixPath('scratch/2305.03393v1-pg9-granite_vision-transformers.html'), PosixPath('scratch/2305.033

## Use Case: Play the Video with Captions in a Browser

Build a minimal HTML page that plays the video with the exported WebVTT
captions overlaid, using the standard `<video>` + `<track>` elements:

In [7]:
video_name = Path(VIDEO_PATH).name
player_html = output_dir / "player.html"
player_html.write_text(f"""<!DOCTYPE html>
<html>
<video controls src="{video_name}">
  <track default kind="captions" srclang="en" label="English" src="video.vtt" />
  Video not supported.
</video>
</html>
""")

print(f"Player page saved to: {player_html}")

Player page saved to: scratch/player.html


> **Note:** Browsers block `<track>` (caption) loading for pages opened
> directly from disk via `file://`. Copy your video into `output/` next
> to `player.html`, then serve the directory and open it over HTTP:
> ```bash
> python -m http.server 8080 --directory output
> ```
> then open `http://localhost:8080/player.html`.
>
> **Firefox:** captions may not appear until you enable them from the
> video's CC/subtitles menu — unlike some browsers, Firefox does not
> always turn on a `default` track automatically.
